# fsaverage → GLB (three.js) + morph metrics + (optional) PyGSP graph

This notebook:
1. Fetches **fsaverage** (high-res) via MNE.
2. Reads the **pial** surface, morph data (**thickness**, **curvature**, **sulc**).
3. Exports a **GLB** (with normals) for three.js.
4. Saves arrays to disk and draws quick histograms for validation.
5. *(Optional)* Builds a **PyGSP** graph from the triangle mesh edges.

> **Tip:** Run left then right hemispheres separately the first time to validate outputs.


In [2]:
# Install required packages (run this first if packages are missing)
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package.split('==')[0])
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Required packages for this notebook
packages = [
    "mne",
    "nibabel", 
    "trimesh",
    "pygsp",
    "scipy",
    "matplotlib"
]

for pkg in packages:
    install_if_missing(pkg)

print("\nPackage installation complete!")

Installing mne...
✓ nibabel already installed
Installing trimesh...
Installing pygsp...
✓ scipy already installed
✓ matplotlib already installed

Package installation complete!


In [ ]:
import os, numpy as np, nibabel as nib, mne, trimesh
from pathlib import Path

OUT = Path(r"C:\CodingProjects\bioctree\external\out"); OUT.mkdir(parents=True, exist_ok=True)
fs_path = mne.datasets.fetch_fsaverage(verbose=True)   # downloads to your Windows user cache
subjects_dir = Path(fs_path).parent
surf_dir = subjects_dir / "fsaverage" / "surf"

hemi = "lh"               # change to "rh" for right hemi
surf = "sphere"           # sphere surface for spherical mapping
sphere_path = surf_dir / f"{hemi}.{surf}"
thick_path  = surf_dir / f"{hemi}.thickness"
curv_path   = surf_dir / f"{hemi}.curv"
sulc_path   = surf_dir / f"{hemi}.sulc"

V, F = nib.freesurfer.read_geometry(sphere_path)         # (N,3), (F,3)
th = nib.freesurfer.read_morph_data(thick_path)
cv = nib.freesurfer.read_morph_data(curv_path)
sl = nib.freesurfer.read_morph_data(sulc_path)

print(f"{hemi}: verts={V.shape[0]:,}, faces={F.shape[0]:,}")
print("thickness:", float(np.nanmin(th)), float(np.nanmedian(th)), float(np.nanmax(th)))
print("curvature:", float(np.nanmin(cv)), float(np.nanmedian(cv)), float(np.nanmax(cv)))
print("sulc:", float(np.nanmin(sl)), float(np.nanmedian(sl)), float(np.nanmax(sl)))

# Create mesh with vertex colors based on curvature (default overlay)
mesh = trimesh.Trimesh(vertices=V.astype(np.float32), faces=F.astype(np.int64), process=False)
mesh.rezero()
mesh.vertex_normals  # This automatically computes normals when accessed

# Export GLB with default overlay (curvature)
glb_path = OUT / f"{hemi}_{surf}_fsaverage.glb"
mesh.export(glb_path.as_posix())
print("GLB:", glb_path)

# Save geometry arrays
np.save(OUT / f"{hemi}_coords.npy", V.astype(np.float32))
np.save(OUT / f"{hemi}_faces.npy",  F.astype(np.int64))

# Save morphometric overlays
np.save(OUT / f"{hemi}_thickness.npy", th.astype(np.float32))
np.save(OUT / f"{hemi}_curvature.npy", cv.astype(np.float32))
np.save(OUT / f"{hemi}_sulc.npy",      sl.astype(np.float32))

print(f"Saved geometry and overlays to {OUT}")

# Create PyGSP graph from mesh
try:
    import pygsp as pg
    from scipy.sparse import coo_matrix, csr_matrix, save_npz
    
    def faces_to_unique_edges(faces):
        """Extract unique edges from triangle faces"""
        f = faces.astype(np.int64)
        e01 = np.sort(f[:, [0,1]], axis=1)
        e12 = np.sort(f[:, [1,2]], axis=1)
        e20 = np.sort(f[:, [2,0]], axis=1)
        edges = np.vstack([e01, e12, e20])
        edges = np.unique(edges, axis=0)
        return edges

    def build_graph(coords, faces, scheme='geodesic'):
        """Build PyGSP graph from mesh"""
        edges = faces_to_unique_edges(faces)
        vi = coords[edges[:,0]]
        vj = coords[edges[:,1]]
        d = np.linalg.norm(vi - vj, axis=1)
        eps = 1e-12

        if scheme == 'unweighted':
            w = np.ones_like(d)
        elif scheme == 'geodesic':
            w = 1.0 / (d + eps)
        elif scheme == 'heat':
            sigma = np.median(d)
            w = np.exp(-(d**2) / (2.0 * sigma**2))
        else:
            raise ValueError('Unknown scheme')

        # Create symmetric adjacency matrix
        i = edges[:,0]; j = edges[:,1]
        data = np.concatenate([w, w])
        row  = np.concatenate([i, j])
        col  = np.concatenate([j, i])
        N = coords.shape[0]
        W = coo_matrix((data, (row, col)), shape=(N, N)).tocsr()

        # Create PyGSP graph
        G = pg.graphs.Graph(W, gtype='mesh')
        G.set_coordinates(coords)
        G.compute_laplacian(lap_type='combinatorial')
        return G, W

    # Build graph with geodesic weighting
    G, W = build_graph(V, F, scheme='geodesic')
    print(f'PyGSP Graph: N={G.N:,}, Edges={G.Ne:,}, Avg degree={2*G.Ne/G.N:.1f}')
    
    # Save graph adjacency matrix
    adjacency_path = OUT / f'{hemi}_adjacency.npz'
    save_npz(adjacency_path, csr_matrix(W))
    print(f'Saved adjacency matrix to {adjacency_path}')
    
    # Save graph coordinates (same as vertices but for consistency)
    np.save(OUT / f"{hemi}_graph_coords.npy", G.coords.astype(np.float32))
    
    # Compute and save graph Laplacian eigenvalues/eigenvectors (first few)
    try:
        G.compute_fourier_basis(n_eigenvectors=min(100, G.N//10))
        np.save(OUT / f"{hemi}_eigenvalues.npy", G.e.astype(np.float32))
        np.save(OUT / f"{hemi}_eigenvectors.npy", G.U.astype(np.float32))
        print(f'Saved graph Fourier basis: {len(G.e)} eigenvalues/eigenvectors')
    except Exception as e:
        print(f'Could not compute Fourier basis: {e}')
        
except ImportError:
    print('PyGSP not available - skipping graph creation')
except Exception as e:
    print(f'PyGSP graph creation failed: {e}')

0 files missing from root.txt in C:\Users\diell\mne_data\MNE-fsaverage-data
0 files missing from bem.txt in C:\Users\diell\mne_data\MNE-fsaverage-data\fsaverage
0 files missing from bem.txt in C:\Users\diell\mne_data\MNE-fsaverage-data\fsaverage
lh: verts=163,842, faces=327,680
thickness: 0.0 2.2264928817749023 4.713199615478516
lh: verts=163,842, faces=327,680
thickness: 0.0 2.2264928817749023 4.713199615478516
GLB: C:\CodingProjects\bioctree\external\out\lh_pial_fsaverage.glb
GLB: C:\CodingProjects\bioctree\external\out\lh_pial_fsaverage.glb


In [ ]:
# Create GLB files with different morphometric overlays as vertex colors
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm

def create_vertex_colors(values, colormap='viridis', vmin=None, vmax=None):
    """Convert scalar values to RGB vertex colors"""
    # Handle NaN values
    valid_mask = ~np.isnan(values)
    if not np.any(valid_mask):
        return np.ones((len(values), 3), dtype=np.float32) * 0.5  # Gray for all NaN
    
    # Set color range
    if vmin is None:
        vmin = np.nanmin(values)
    if vmax is None:
        vmax = np.nanmax(values)
    
    # Normalize values
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.get_cmap(colormap)
    
    # Create colors
    colors = np.ones((len(values), 3), dtype=np.float32) * 0.5  # Default gray
    colors[valid_mask] = cmap(norm(values[valid_mask]))[:, :3]  # RGB only
    
    return colors

# Create GLB files for each overlay
overlays = {
    'thickness': th,
    'curvature': cv, 
    'sulc': sl
}

colormaps = {
    'thickness': 'plasma',    # Hot colors for thickness
    'curvature': 'RdBu_r',    # Red-blue for curvature (+ convex, - concave)
    'sulc': 'coolwarm'        # Cool-warm for sulcal depth
}

print("Creating GLB files with morphometric overlays...")

for overlay_name, overlay_data in overlays.items():
    print(f"\nProcessing {overlay_name} overlay...")
    
    # Create vertex colors
    vertex_colors = create_vertex_colors(
        overlay_data, 
        colormap=colormaps[overlay_name],
        vmin=np.nanpercentile(overlay_data, 2),  # Robust range
        vmax=np.nanpercentile(overlay_data, 98)
    )
    
    # Create mesh with colored vertices
    colored_mesh = trimesh.Trimesh(
        vertices=V.astype(np.float32), 
        faces=F.astype(np.int64), 
        vertex_colors=vertex_colors,
        process=False
    )
    colored_mesh.rezero()
    colored_mesh.vertex_normals  # Compute normals
    
    # Export GLB
    overlay_glb_path = OUT / f"{hemi}_{surf}_{overlay_name}_fsaverage.glb"
    colored_mesh.export(overlay_glb_path.as_posix())
    
    print(f"  Exported: {overlay_glb_path}")
    print(f"  Value range: {np.nanmin(overlay_data):.3f} to {np.nanmax(overlay_data):.3f}")
    print(f"  Color range: {np.nanpercentile(overlay_data, 2):.3f} to {np.nanpercentile(overlay_data, 98):.3f}")

print(f"\nAll GLB files saved to {OUT}")
print("Files created:")
print(f"  - {hemi}_{surf}_fsaverage.glb (basic mesh)")
print(f"  - {hemi}_{surf}_thickness_fsaverage.glb (thickness overlay)")
print(f"  - {hemi}_{surf}_curvature_fsaverage.glb (curvature overlay)")
print(f"  - {hemi}_{surf}_sulc_fsaverage.glb (sulcal depth overlay)")

In [5]:
# Environment check (run this first)
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

import numpy as np, nibabel as nib, mne, trimesh
print('numpy:', np.__version__)
print('nibabel:', nib.__version__)
print('mne:', mne.__version__)
print('trimesh:', trimesh.__version__)

try:
    import pygsp as pg
    print('pygsp:', pg.__version__)
except Exception as e:
    print('pygsp not found or failed to import:', e)

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
Platform: Windows-11-10.0.26100-SP0
numpy: 2.3.4
nibabel: 5.3.2
mne: 1.10.2
trimesh: 4.9.0
pygsp: 0.6.1


In [ ]:
# Settings
OUTDIR = './out'      # artifacts will be written here
HEMI = 'lh'           # 'lh' or 'rh'
SURFACE = 'sphere'    # 'sphere' for spherical surface (was 'pial' or 'white')

# Create output directory if needed
import os
os.makedirs(OUTDIR, exist_ok=True)
print('Output dir:', OUTDIR)
print('Processing sphere surface for spherical mapping and graph analysis')

In [ ]:
# Fetch the FreeSurfer fsaverage subject tree (includes surfaces & morph maps)
fs_path = mne.datasets.fetch_fsaverage(verbose=True)
import os
subjects_dir = os.path.dirname(fs_path)  # parent of 'fsaverage'
subject = 'fsaverage'
surf_dir = os.path.join(subjects_dir, subject, 'surf')

print('subjects_dir:', subjects_dir)
print('surf_dir:', surf_dir)

In [ ]:
# Read surface geometry and per-vertex morph measures
import os
from pathlib import Path
import numpy as np
import nibabel as nib

sphere_path = os.path.join(surf_dir, f'{HEMI}.{SURFACE}')   # sphere surface
thick_path = os.path.join(surf_dir, f'{HEMI}.thickness')   # cortical thickness
curv_path  = os.path.join(surf_dir, f'{HEMI}.curv')        # mean curvature
sulc_path  = os.path.join(surf_dir, f'{HEMI}.sulc')        # sulcal depth

coords, faces = nib.freesurfer.read_geometry(sphere_path)   # (N,3), (F,3) - sphere coords
thickness = nib.freesurfer.read_morph_data(thick_path)     # (N,)
curvature = nib.freesurfer.read_morph_data(curv_path)      # (N,)
sulc      = nib.freesurfer.read_morph_data(sulc_path)      # (N,)

coords = coords.astype('float32')
faces  = faces.astype('int64')

print(f'{HEMI} sphere: verts={coords.shape[0]:,}, faces={faces.shape[0]:,}')
print('thickness stats: min=%.3f med=%.3f max=%.3f' % (thickness.min(), np.median(thickness), thickness.max()))
print('curvature stats: min=%.3f med=%.3f max=%.3f' % (curvature.min(), np.median(curvature), curvature.max()))
print('sulc stats:      min=%.3f med=%.3f max=%.3f' % (sulc.min(), np.median(sulc), sulc.max()))

# Verify sphere properties
sphere_radius = np.linalg.norm(coords, axis=1)
print(f'Sphere radius stats: min=%.2f med=%.2f max=%.2f' % (sphere_radius.min(), np.median(sphere_radius), sphere_radius.max()))

In [ ]:
# Export GLB with normals (three.js-ready sphere)
import trimesh
from pathlib import Path

mesh = trimesh.Trimesh(vertices=coords, faces=faces, process=False)
mesh.rezero()
mesh.vertex_normals  # Compute vertex normals

glb_path = Path(OUTDIR) / f'{HEMI}_{SURFACE}_fsaverage.glb'
mesh.export(glb_path.as_posix())

print(f'Exported sphere GLB: {glb_path}')
print(f'Sphere center: [{np.mean(coords, axis=0)}]')
print(f'Sphere radius range: {np.linalg.norm(coords, axis=1).min():.2f} - {np.linalg.norm(coords, axis=1).max():.2f}')

glb_path

In [ ]:
# Save arrays to disk for later validation/processing
import numpy as np
from pathlib import Path

np.save(Path(OUTDIR) / f'{HEMI}_coords.npy', coords)
np.save(Path(OUTDIR) / f'{HEMI}_faces.npy', faces)
np.save(Path(OUTDIR) / f'{HEMI}_thickness.npy', thickness.astype('float32'))
np.save(Path(OUTDIR) / f'{HEMI}_curvature.npy', curvature.astype('float32'))
np.save(Path(OUTDIR) / f'{HEMI}_sulc.npy',      sulc.astype('float32'))

print('Saved coords/faces/thickness/curvature/sulc to', OUTDIR)

In [ ]:
# Quick visual validation (histograms) – one chart per metric
# NOTE: Per policy, use matplotlib (no seaborn), one chart per figure, no specific colors.
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.hist(thickness[~np.isnan(thickness)], bins=100)
plt.title(f'{HEMI} thickness histogram')
plt.xlabel('Thickness'); plt.ylabel('Count')
plt.show()

plt.figure()
plt.hist(curvature[~np.isnan(curvature)], bins=100)
plt.title(f'{HEMI} curvature histogram')
plt.xlabel('Curvature'); plt.ylabel('Count')
plt.show()

plt.figure()
plt.hist(sulc[~np.isnan(sulc)], bins=100)
plt.title(f'{HEMI} sulc histogram')
plt.xlabel('Sulc'); plt.ylabel('Count')
plt.show()

## Optional: Build a PyGSP graph from the mesh

This uses triangle edges as undirected graph edges and assigns weights based on edge lengths
(you can switch to unweighted or heat-kernel style if preferred).

In [ ]:
# Build PyGSP graph from sphere mesh
try:
    import numpy as np
    import pygsp as pg
    from scipy.sparse import coo_matrix, csr_matrix, save_npz
    from pathlib import Path

    def faces_to_unique_edges(faces):
        """Extract unique edges from triangle faces"""
        f = faces.astype(np.int64)
        e01 = np.sort(f[:, [0,1]], axis=1)
        e12 = np.sort(f[:, [1,2]], axis=1)
        e20 = np.sort(f[:, [2,0]], axis=1)
        edges = np.vstack([e01, e12, e20])
        edges = np.unique(edges, axis=0)
        return edges

    def build_graph(coords, faces, scheme='geodesic'):
        """Build PyGSP graph from mesh with various weighting schemes"""
        edges = faces_to_unique_edges(faces)
        vi = coords[edges[:,0]]
        vj = coords[edges[:,1]]
        d = np.linalg.norm(vi - vj, axis=1)
        eps = 1e-12

        if scheme == 'unweighted':
            w = np.ones_like(d)
        elif scheme == 'geodesic':
            w = 1.0 / (d + eps)
        elif scheme == 'heat':
            sigma = np.median(d)
            w = np.exp(-(d**2) / (2.0 * sigma**2))
        elif scheme == 'spherical':
            # For sphere, use great circle distance
            # Normalize to unit sphere first
            vi_norm = vi / np.linalg.norm(vi, axis=1, keepdims=True)
            vj_norm = vj / np.linalg.norm(vj, axis=1, keepdims=True)
            # Great circle distance
            dot_prod = np.sum(vi_norm * vj_norm, axis=1)
            dot_prod = np.clip(dot_prod, -1, 1)  # Handle numerical errors
            spherical_dist = np.arccos(dot_prod)
            w = 1.0 / (spherical_dist + eps)
        else:
            raise ValueError('Unknown scheme')

        # Create symmetric adjacency matrix
        i = edges[:,0]; j = edges[:,1]
        data = np.concatenate([w, w])
        row  = np.concatenate([i, j])
        col  = np.concatenate([j, i])
        N = coords.shape[0]
        W = coo_matrix((data, (row, col)), shape=(N, N)).tocsr()

        # Create PyGSP graph
        G = pg.graphs.Graph(W, gtype='sphere' if 'spherical' in scheme else 'mesh')
        G.set_coordinates(coords)
        G.compute_laplacian(lap_type='combinatorial')
        return G, W

    print("Building PyGSP graphs with different weighting schemes...")
    
    schemes = ['geodesic', 'spherical', 'heat']
    graphs = {}
    
    for scheme in schemes:
        print(f"\nBuilding {scheme} weighted graph...")
        G, W = build_graph(coords, faces, scheme=scheme)
        graphs[scheme] = {'graph': G, 'adjacency': W}
        
        print(f'  Graph: N={G.N:,}, Edges={G.Ne:,}, Avg degree={2*G.Ne/G.N:.1f}')
        
        # Save adjacency matrix
        adj_path = Path(OUTDIR) / f'{HEMI}_{scheme}_adjacency.npz'
        save_npz(adj_path, csr_matrix(W))
        print(f'  Saved adjacency: {adj_path}')
        
        # Compute and save Fourier basis (first 100 eigenvectors)
        try:
            n_eigs = min(100, G.N//10)
            G.compute_fourier_basis(n_eigenvectors=n_eigs)
            
            eig_vals_path = Path(OUTDIR) / f'{HEMI}_{scheme}_eigenvalues.npy'
            eig_vecs_path = Path(OUTDIR) / f'{HEMI}_{scheme}_eigenvectors.npy'
            
            np.save(eig_vals_path, G.e.astype(np.float32))
            np.save(eig_vecs_path, G.U.astype(np.float32))
            
            print(f'  Saved {len(G.e)} eigenvalues/eigenvectors')
            print(f'    Eigenvalue range: {G.e.min():.6f} - {G.e.max():.2f}')
            
        except Exception as e:
            print(f'  Could not compute Fourier basis: {e}')
    
    # Save graph coordinates (sphere coordinates)
    coords_path = Path(OUTDIR) / f'{HEMI}_sphere_coords.npy'
    np.save(coords_path, coords.astype(np.float32))
    print(f'\nSaved sphere coordinates: {coords_path}')
    
    # Additional sphere-specific analysis
    print(f"\nSphere analysis:")
    radii = np.linalg.norm(coords, axis=1)
    print(f"  Radius variation: {radii.std():.6f} (should be small for good sphere)")
    print(f"  Mean radius: {radii.mean():.2f}")
    
    # Check if it's approximately centered
    center = np.mean(coords, axis=1)
    print(f"  Center offset from origin: {np.linalg.norm(center):.6f}")
        
except ImportError:
    print('PyGSP not available - install with: pip install pygsp')
except Exception as e:
    print(f'PyGSP graph creation failed: {e}')
    import traceback
    traceback.print_exc()

## Summary of Generated Files

This notebook creates the following files for the **sphere surface** with morphometric overlays:

### GLB Files (Three.js ready)
- `lh_sphere_fsaverage.glb` - Basic sphere mesh
- `lh_sphere_thickness_fsaverage.glb` - Sphere with thickness overlay colors
- `lh_sphere_curvature_fsaverage.glb` - Sphere with curvature overlay colors  
- `lh_sphere_sulc_fsaverage.glb` - Sphere with sulcal depth overlay colors

### Geometry Data
- `lh_coords.npy` - Vertex coordinates (sphere coordinates)
- `lh_faces.npy` - Triangle faces
- `lh_sphere_coords.npy` - Same as coords (for consistency)

### Morphometric Overlays
- `lh_thickness.npy` - Cortical thickness values
- `lh_curvature.npy` - Mean curvature values
- `lh_sulc.npy` - Sulcal depth values

### PyGSP Graph Data
- `lh_geodesic_adjacency.npz` - Adjacency matrix (geodesic weighting)
- `lh_spherical_adjacency.npz` - Adjacency matrix (spherical distance weighting)
- `lh_heat_adjacency.npz` - Adjacency matrix (heat kernel weighting)
- `lh_*_eigenvalues.npy` - Graph Laplacian eigenvalues (first 100)
- `lh_*_eigenvectors.npy` - Graph Laplacian eigenvectors (first 100)

### Next Steps
1. **Right hemisphere**: Change `hemi = "rh"` and re-run
2. **Three.js**: Load GLB files with `GLTFLoader`
3. **Graph analysis**: Use PyGSP eigenvalues/eigenvectors for spectral analysis
4. **Visualization**: Sphere provides perfect spherical mapping for data visualization